<a href="https://colab.research.google.com/github/hyunkyung31/coronary-ai-ml-dl/blob/main/hyunkyung/10_AngioCAD_XCA_Clinical_Fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 10_AngioCAD_Clinical_XCA_Fusion
# Cell 1. 새로운 개인 작업 경로 연결 및 필수 파일 Gate
# ============================================================

from pathlib import Path

MY_DRIVE = Path("/content/drive/MyDrive")

# research-data 프로젝트명의 한글 Unicode 차이를 피하기 위한 자동 탐색
work_candidates = []

for project_dir in MY_DRIVE.iterdir():
    candidate = (
        project_dir
        / "03_data"
        / "AngioCAD"
        / "hyunkyung"
    )

    if (
        project_dir.name.startswith("research-data")
        and candidate.exists()
    ):
        work_candidates.append(candidate)

assert len(work_candidates) == 1, (
    "hyunkyung 작업 폴더를 하나로 결정할 수 없습니다.\n"
    f"발견된 경로: {work_candidates}"
)

WORK_ROOT = work_candidates[0]

print("WORK_ROOT:")
print(WORK_ROOT)


# ------------------------------------------------------------
# 현재 폴더 구조 출력
# ------------------------------------------------------------
print("\n=== hyunkyung 하위 폴더 ===")

for path in sorted(WORK_ROOT.iterdir()):
    kind = "DIR " if path.is_dir() else "FILE"
    print(f"[{kind}] {path.name}")


# ------------------------------------------------------------
# Fusion 필수 파일 자동 탐색
# ------------------------------------------------------------
eligible_candidates = list(
    WORK_ROOT.rglob("03_eligible_frame_manifest.csv")
)

clinical_oof_candidates = list(
    WORK_ROOT.rglob("05_oof_predictions.csv")
)

split_candidates = (
    list(WORK_ROOT.rglob("02_patient_split_manifest.csv"))
    + list(WORK_ROOT.rglob("01_patient_split_candidate_seed42.csv"))
)

xca_checkpoint_candidates = (
    list(WORK_ROOT.rglob("best_resnet50_3bin.pt"))
    + list(WORK_ROOT.rglob("best_resnet50.pt"))
)


def print_candidates(title, candidates):
    print(f"\n=== {title} ===")

    if not candidates:
        print("찾지 못함")
        return

    for path in sorted(set(candidates)):
        print(path)


print_candidates(
    "Eligible-frame manifest",
    eligible_candidates,
)

print_candidates(
    "Clinical OOF",
    clinical_oof_candidates,
)

print_candidates(
    "Patient split",
    split_candidates,
)

print_candidates(
    "기존 XCA checkpoint",
    xca_checkpoint_candidates,
)


# ------------------------------------------------------------
# 필수 Gate
# ------------------------------------------------------------
assert len(eligible_candidates) == 1, (
    "03_eligible_frame_manifest.csv가 정확히 하나여야 합니다."
)

assert len(clinical_oof_candidates) == 1, (
    "05_oof_predictions.csv가 정확히 하나여야 합니다."
)

assert len(split_candidates) >= 1, (
    "환자 split manifest를 찾지 못했습니다."
)

ELIGIBLE_MANIFEST = eligible_candidates[0]
CLINICAL_OOF_PATH = clinical_oof_candidates[0]

# 공식 split manifest를 우선 사용
official_split = [
    path
    for path in split_candidates
    if path.name == "02_patient_split_manifest.csv"
]

SPLIT_MANIFEST = (
    official_split[0]
    if official_split
    else split_candidates[0]
)


# ------------------------------------------------------------
# 앞으로의 Fusion 저장 경로
# ------------------------------------------------------------
FUSION_DIR = (
    WORK_ROOT
    / "fusion_results"
    / "10_clinical_xca_fusion"
)

FUSION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("\n=== 확정 경로 ===")
print("Eligible manifest:", ELIGIBLE_MANIFEST)
print("Clinical OOF     :", CLINICAL_OOF_PATH)
print("Split manifest   :", SPLIT_MANIFEST)
print("Fusion output    :", FUSION_DIR)

print("\n✅ 새 폴더 구조 연결 Gate 통과")
print("🔒 TEST 예측 및 평가는 수행하지 않았습니다.")

In [ ]:
# 이동된 Clinical OOF 경로 다시 탐색

clinical_oof_candidates = list(
    WORK_ROOT.rglob("05_oof_predictions.csv")
)

print("발견된 Clinical OOF:")
for path in clinical_oof_candidates:
    print(path)

assert len(clinical_oof_candidates) == 1, (
    "Clinical OOF가 정확히 하나여야 합니다.\n"
    f"발견된 경로: {clinical_oof_candidates}"
)

CLINICAL_OOF_PATH = clinical_oof_candidates[0]

print("\n확정 경로:")
print(CLINICAL_OOF_PATH)

assert CLINICAL_OOF_PATH.exists()

print("\n✅ 이동된 Clinical OOF 경로 재연결 완료")

In [ ]:
# ============================================================
# Cell 2. Multimodal DEVELOPMENT cohort 구성
# - AdaBoost Clinical OOF 303명 기준
# - 각 XCA series에서 middle frame 1장 선택
# - TEST 제외
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. 데이터 불러오기
# ------------------------------------------------------------
clinical_oof = pd.read_csv(CLINICAL_OOF_PATH)
frame_manifest = pd.read_csv(ELIGIBLE_MANIFEST)
split_manifest = pd.read_csv(SPLIT_MANIFEST)

print("Clinical OOF:", clinical_oof.shape)
print("Frame manifest:", frame_manifest.shape)
print("Split manifest:", split_manifest.shape)

print("\nSplit columns:")
print(split_manifest.columns.tolist())


# ------------------------------------------------------------
# 2. patient_id 정규화
# ------------------------------------------------------------
def normalize_patient_id(series):
    converted = pd.to_numeric(
        series,
        errors="raise",
    )

    return converted.astype("int64")


for dataframe in [
    clinical_oof,
    frame_manifest,
    split_manifest,
]:
    dataframe["patient_id"] = normalize_patient_id(
        dataframe["patient_id"]
    )


# ------------------------------------------------------------
# 3. Primary Clinical model: AdaBoost OOF
# ------------------------------------------------------------
clinical_primary = (
    clinical_oof[
        clinical_oof["model"].eq("adaboost")
    ]
    [
        [
            "patient_id",
            "fold",
            "target",
            "probability",
            "prediction",
        ]
    ]
    .rename(
        columns={
            "probability": "clinical_probability",
            "prediction": "clinical_prediction",
        }
    )
    .copy()
)

assert len(clinical_primary) == 303
assert clinical_primary["patient_id"].nunique() == 303
assert not clinical_primary["patient_id"].duplicated().any()
assert clinical_primary["fold"].between(0, 4).all()
assert set(clinical_primary["target"].unique()) == {0, 1}

print("\n=== Clinical primary OOF ===")
print("Patients:", clinical_primary["patient_id"].nunique())
print(
    "Target:",
    clinical_primary["target"]
    .value_counts()
    .sort_index()
    .to_dict(),
)
print(
    "Fold:",
    clinical_primary["fold"]
    .value_counts()
    .sort_index()
    .to_dict(),
)


# ------------------------------------------------------------
# 4. paper_eligible frame만 사용
# ------------------------------------------------------------
def to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )


frame_manifest["paper_eligible"] = to_bool(
    frame_manifest["paper_eligible"]
)

paper_frames = (
    frame_manifest[
        frame_manifest["paper_eligible"]
    ]
    .copy()
)

paper_frames["frame_index"] = pd.to_numeric(
    paper_frames["frame_index"],
    errors="raise",
).astype("int64")

paper_frames["series_frame_count"] = pd.to_numeric(
    paper_frames["series_frame_count"],
    errors="raise",
).astype("int64")


# ------------------------------------------------------------
# 5. 각 patient-series에서 중앙에 가장 가까운 frame 1장 선택
# ------------------------------------------------------------
paper_frames["middle_position"] = (
    paper_frames["series_frame_count"] - 1
) / 2.0

paper_frames["middle_distance"] = (
    paper_frames["frame_index"]
    - paper_frames["middle_position"]
).abs()

series_keys = [
    "patient_id",
    "series_id",
]

middle_frames = (
    paper_frames
    .sort_values(
        series_keys
        + [
            "middle_distance",
            "frame_index",
        ]
    )
    .drop_duplicates(
        subset=series_keys,
        keep="first",
    )
    .copy()
)

assert not middle_frames.duplicated(series_keys).any()

print("\n=== Paper-eligible middle frames ===")
print("Series:", len(middle_frames))
print("Patients:", middle_frames["patient_id"].nunique())
print("\nView distribution:")
print(middle_frames["view"].value_counts(dropna=False))


# ------------------------------------------------------------
# 6. Clinical 303명과 XCA 결합
# ------------------------------------------------------------
multimodal_series = middle_frames.merge(
    clinical_primary,
    on="patient_id",
    how="inner",
    validate="many_to_one",
)

clinical_patient_ids = set(
    clinical_primary["patient_id"]
)

xca_patient_ids = set(
    multimodal_series["patient_id"]
)

missing_xca_patients = sorted(
    clinical_patient_ids - xca_patient_ids
)

print("\n=== Multimodal overlap ===")
print("Clinical patients:", len(clinical_patient_ids))
print("XCA-overlap patients:", len(xca_patient_ids))
print("XCA가 없는 Clinical patients:", len(missing_xca_patients))

if missing_xca_patients:
    print("Missing patient IDs:")
    print(missing_xca_patients)


# ------------------------------------------------------------
# 7. 환자별 series 개수 확인
# ------------------------------------------------------------
patient_series_count = (
    multimodal_series
    .groupby(
        [
            "patient_id",
            "fold",
            "target",
        ],
        as_index=False,
    )
    .agg(
        series_count=("series_id", "nunique"),
        right_series_count=(
            "view",
            lambda values: (
                values.astype(str).str.upper() == "RIGHT"
            ).sum(),
        ),
        left_series_count=(
            "view",
            lambda values: (
                values.astype(str).str.upper() == "LEFT"
            ).sum(),
        ),
    )
)

print("\n=== Patient series-count summary ===")
display(
    patient_series_count[
        [
            "series_count",
            "right_series_count",
            "left_series_count",
        ]
    ].describe()
)

print("\n=== Fold × target ===")
display(
    pd.crosstab(
        patient_series_count["fold"],
        patient_series_count["target"],
        margins=True,
    )
)


# ------------------------------------------------------------
# 8. Colab 현재 세션에서 이미지 경로가 유효한지 확인
# ------------------------------------------------------------
multimodal_series["file_exists"] = (
    multimodal_series["file_path"]
    .map(lambda path: Path(str(path)).exists())
)

existing_count = int(
    multimodal_series["file_exists"].sum()
)

print("\n=== Image path check ===")
print("Selected series:", len(multimodal_series))
print("Existing image files:", existing_count)
print(
    "Missing image files:",
    len(multimodal_series) - existing_count,
)

if existing_count == 0:
    print(
        "\n⚠️ Manifest의 /content 경로에 영상이 없습니다."
        "\n다음 단계에서 AngioCAD 영상 압축을 다시 연결하거나"
        " file_path를 갱신해야 합니다."
    )
elif existing_count < len(multimodal_series):
    print(
        "\n⚠️ 일부 이미지 경로만 존재합니다."
        "\n누락 경로를 확인해야 합니다."
    )


# ------------------------------------------------------------
# 9. Cohort Gate
# ------------------------------------------------------------
assert not missing_xca_patients, (
    "Clinical 환자 중 paper-eligible XCA가 없는 환자가 있습니다. "
    "교집합 cohort 재정의가 필요합니다."
)

assert patient_series_count["patient_id"].nunique() == 303
assert not patient_series_count["patient_id"].duplicated().any()

assert (
    patient_series_count
    .groupby("fold")["target"]
    .nunique()
    .eq(2)
    .all()
), "일부 fold에 한쪽 target만 존재합니다."


# ------------------------------------------------------------
# 10. Fusion 입력 manifest 저장
# ------------------------------------------------------------
series_output_path = (
    FUSION_DIR
    / "00_multimodal_middle_frame_manifest.csv"
)

patient_output_path = (
    FUSION_DIR
    / "01_multimodal_patient_cohort.csv"
)

multimodal_series.to_csv(
    series_output_path,
    index=False,
)

patient_series_count.to_csv(
    patient_output_path,
    index=False,
)

print("\n저장:", series_output_path)
print("저장:", patient_output_path)

print("\n✅ Multimodal DEVELOPMENT cohort Gate 통과")
print("✅ Clinical 303명과 XCA series 연결 완료")
print("🔒 TEST 환자는 포함하거나 평가하지 않았습니다.")

In [ ]:
# ============================================================
# Cell 3. AngioCAD 원본 영상/압축파일 위치 탐색
# - read-only
# - 아직 압축을 풀거나 파일을 이동하지 않음
# ============================================================

from pathlib import Path
import os

# WORK_ROOT:
# .../research-data/03_data/AngioCAD/hyunkyung

PROJECT_ROOT = WORK_ROOT.parents[2]
DATA_ROOT = WORK_ROOT.parent

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT   :", DATA_ROOT)


# ------------------------------------------------------------
# 1. Colab 로컬 런타임 후보
# ------------------------------------------------------------
local_candidates = [
    Path("/content/AngioCAD/AngioCAD_Dataset"),
    Path("/content/AngioCAD_Dataset"),
    Path("/content/AngioCAD"),
]

print("\n=== Colab local candidates ===")

for path in local_candidates:
    print(
        f"{path} | "
        f"exists={path.exists()} | "
        f"is_dir={path.is_dir()}"
    )


# ------------------------------------------------------------
# 2. 프로젝트 내부 AngioCAD_Dataset 폴더 및 압축파일 탐색
# ------------------------------------------------------------
dataset_directories = []
archive_candidates = []

archive_suffixes = (
    ".zip",
    ".tar",
    ".tar.gz",
    ".tgz",
    ".7z",
    ".rar",
)

for current_root, directory_names, file_names in os.walk(PROJECT_ROOT):
    current_path = Path(current_root)

    # 실제 영상 디렉터리를 발견하면 내부 수십만 파일까지 탐색하지 않음
    if current_path.name == "AngioCAD_Dataset":
        dataset_directories.append(current_path)
        directory_names[:] = []
        continue

    if "AngioCAD_Dataset" in directory_names:
        dataset_path = current_path / "AngioCAD_Dataset"
        dataset_directories.append(dataset_path)

        # 해당 대용량 폴더 내부 탐색 제외
        directory_names.remove("AngioCAD_Dataset")

    for filename in file_names:
        lower_name = filename.lower()

        if (
            "angio" in lower_name
            and lower_name.endswith(archive_suffixes)
        ):
            archive_candidates.append(
                current_path / filename
            )


print("\n=== Drive의 AngioCAD_Dataset 폴더 ===")

if dataset_directories:
    for path in sorted(set(dataset_directories)):
        print(path)
else:
    print("찾지 못함")


print("\n=== AngioCAD 압축파일 후보 ===")

if archive_candidates:
    for path in sorted(set(archive_candidates)):
        size_gb = path.stat().st_size / (1024 ** 3)

        print(
            f"{path}\n"
            f"  size={size_gb:.2f} GB"
        )
else:
    print("찾지 못함")


# ------------------------------------------------------------
# 3. external_data 바로 아래 구조 확인
# ------------------------------------------------------------
external_dir = WORK_ROOT / "external_data"

print("\n=== external_data 상위 구조 ===")

if external_dir.exists():
    for path in sorted(external_dir.iterdir()):
        if path.is_file():
            size_gb = path.stat().st_size / (1024 ** 3)
            print(f"[FILE] {path.name} | {size_gb:.2f} GB")
        else:
            print(f"[DIR ] {path.name}")
else:
    print("external_data 폴더 없음")


# ------------------------------------------------------------
# 4. Manifest가 기대하는 실제 경로 예시
# ------------------------------------------------------------
saved_manifest = pd.read_csv(
    FUSION_DIR
    / "00_multimodal_middle_frame_manifest.csv"
)

print("\n=== Manifest 경로 예시 ===")
print(saved_manifest["file_path"].iloc[0])

print("\n✅ 원본 영상 위치 탐색 완료")

- 환자 1명을 하나의 학습 샘플로 취급
- 환자당 2~20개의 middle frame을 모두 입력
- ResNet50으로 각 frame feature 추출
- LEFT/RIGHT view 정보를 함께 입력
- attention MIL로 환자 단위 XCA 점수 생성
- 고정된 5-fold에서 환자별 OOF 예측 생성
- AdaBoost Clinical OOF와 1:1 평균 fusion
- fusion 가중치는 결과를 보고 조정하지 않음
- 기존 08b checkpoint는 fold 0 실험이므로 5-fold - OOF 초기값으로 사용하지 않음
- TEST 미사용

In [ ]:
# ============================================================
# 완전 독립 실행 셀: Drive 연결 → RAR 탐색 → 압축 해제 → 검증
# 이전 셀의 변수에 전혀 의존하지 않음
# ============================================================

from google.colab import drive
from pathlib import Path
import pandas as pd
import shutil
import subprocess
import re
import torch

drive.mount("/content/drive", force_remount=False)


# ------------------------------------------------------------
# 1. 프로젝트 경로 자동 탐색
# ------------------------------------------------------------
MY_DRIVE = Path("/content/drive/MyDrive")

project_candidates = []

for path in MY_DRIVE.iterdir():
    if (
        path.name.startswith("research-data")
        and (path / "03_data" / "AngioCAD").exists()
    ):
        project_candidates.append(path)

assert len(project_candidates) == 1, (
    f"프로젝트 경로를 하나로 결정할 수 없습니다: "
    f"{project_candidates}"
)

PROJECT_ROOT = project_candidates[0]
DATA_ROOT = PROJECT_ROOT / "03_data" / "AngioCAD"
WORK_ROOT = DATA_ROOT / "hyunkyung"

FUSION_DIR = (
    WORK_ROOT
    / "fusion_results"
    / "10_clinical_xca_fusion"
)

assert DATA_ROOT.exists(), DATA_ROOT
assert WORK_ROOT.exists(), WORK_ROOT
assert FUSION_DIR.exists(), FUSION_DIR

print("DATA_ROOT:", DATA_ROOT)
print("WORK_ROOT:", WORK_ROOT)
print("GPU:", (
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
))

assert torch.cuda.is_available(), (
    "현재 GPU 런타임이 아닙니다."
)


# ------------------------------------------------------------
# 2. Multimodal manifest 자동 탐색
# ------------------------------------------------------------
manifest_candidates = list(
    FUSION_DIR.rglob(
        "00_multimodal_middle_frame_manifest.csv"
    )
)

assert len(manifest_candidates) == 1, (
    f"Multimodal manifest를 하나로 결정할 수 없습니다: "
    f"{manifest_candidates}"
)

MANIFEST_PATH = manifest_candidates[0]

print("Manifest:", MANIFEST_PATH)


# ------------------------------------------------------------
# 3. RAR part 자동 탐색
# ------------------------------------------------------------
rar_parts = list(
    DATA_ROOT.glob("AngioCAD_Dataset.part*.rar")
)

rar_parts = sorted(
    rar_parts,
    key=lambda path: int(
        re.search(
            r"\.part(\d+)\.rar$",
            path.name,
        ).group(1)
    ),
)

part_numbers = [
    int(
        re.search(
            r"\.part(\d+)\.rar$",
            path.name,
        ).group(1)
    )
    for path in rar_parts
]

assert part_numbers == [1, 2, 3, 4], (
    f"RAR part 구성이 잘못됐습니다: {part_numbers}"
)

print("\n=== 발견된 RAR ===")
for path in rar_parts:
    print(
        path.name,
        f"{path.stat().st_size / (1024**3):.2f} GB",
    )


# ------------------------------------------------------------
# 4. 로컬 디스크 확인
# ------------------------------------------------------------
disk = shutil.disk_usage("/content")
free_gb = disk.free / (1024**3)

print(f"\n/content 여유 공간: {free_gb:.2f} GB")

assert free_gb >= 60, (
    "압축 해제를 위한 로컬 공간이 부족합니다."
)


# ------------------------------------------------------------
# 5. RAR 4개를 로컬로 복사
# ------------------------------------------------------------
LOCAL_ARCHIVE_DIR = Path(
    "/content/AngioCAD_archives"
)
LOCAL_ARCHIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

local_parts = []

for source in rar_parts:
    destination = (
        LOCAL_ARCHIVE_DIR
        / source.name
    )

    if (
        destination.exists()
        and destination.stat().st_size
        == source.stat().st_size
    ):
        print(f"✅ 복사 생략: {destination.name}")
    else:
        print(f"\n복사 시작: {source.name}")

        shutil.copy2(
            source,
            destination,
        )

        assert (
            destination.stat().st_size
            == source.stat().st_size
        )

        print(f"✅ 복사 완료: {destination.name}")

    local_parts.append(destination)

LOCAL_PART1 = local_parts[0]


# ------------------------------------------------------------
# 6. unar 설치
# ------------------------------------------------------------
if shutil.which("unar") is None:
    print("\nunar 설치 중...")

    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True,
    )

    subprocess.run(
        [
            "apt-get",
            "install",
            "-y",
            "-qq",
            "unar",
        ],
        check=True,
    )

assert shutil.which("unar") is not None
print("\nunar:", shutil.which("unar"))


# ------------------------------------------------------------
# 7. 압축 해제
# ------------------------------------------------------------
EXTRACT_ROOT = Path(
    "/content/AngioCAD_unar"
)
EXTRACT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("\n=== 압축 해제 시작 ===")
print("입력:", LOCAL_PART1)
print("출력:", EXTRACT_ROOT)

process = subprocess.Popen(
    [
        "unar",
        "-f",
        "-o",
        str(EXTRACT_ROOT),
        str(LOCAL_PART1),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line.rstrip())

return_code = process.wait()

assert return_code == 0, (
    f"unar 압축 해제 실패: {return_code}"
)

print("\n✅ 압축 해제 완료")


# ------------------------------------------------------------
# 8. 실제 AngioCAD_Dataset 위치 탐색
# ------------------------------------------------------------
dataset_roots = [
    path
    for path in EXTRACT_ROOT.rglob(
        "AngioCAD_Dataset"
    )
    if path.is_dir()
]

print("\n=== Dataset root 후보 ===")
for path in dataset_roots:
    print(path)

assert len(dataset_roots) == 1, (
    f"Dataset root를 하나로 결정할 수 없습니다: "
    f"{dataset_roots}"
)

ACTUAL_DATASET_ROOT = dataset_roots[0]


# ------------------------------------------------------------
# 9. Manifest 경로 갱신
# ------------------------------------------------------------
manifest = pd.read_csv(
    MANIFEST_PATH
)


def rebuild_path(old_path):
    old_path = Path(str(old_path))
    parts = old_path.parts

    assert "AngioCAD_Dataset" in parts

    index = parts.index(
        "AngioCAD_Dataset"
    )

    relative_path = Path(
        *parts[index + 1:]
    )

    return str(
        ACTUAL_DATASET_ROOT
        / relative_path
    )


manifest["file_path"] = (
    manifest["file_path"]
    .map(rebuild_path)
)

manifest["file_exists"] = (
    manifest["file_path"]
    .map(
        lambda value: Path(value).exists()
    )
)

existing = int(
    manifest["file_exists"].sum()
)
missing = len(manifest) - existing

print("\n=== 영상 Gate ===")
print("전체:", len(manifest))
print("존재:", existing)
print("누락:", missing)

if missing:
    print(
        manifest.loc[
            ~manifest["file_exists"],
            "file_path",
        ]
        .head(10)
        .to_string(index=False)
    )

assert missing == 0, (
    "일부 middle frame을 찾지 못했습니다."
)

manifest.to_csv(
    MANIFEST_PATH,
    index=False,
)

print("\n✅ XCA 영상 1,973개 복원 완료")
print("✅ GPU 확인 완료")
print("이제 Cell 5부터 진행하면 됩니다.")

In [ ]:
# ============================================================
# Cell 5. XCA patient-level MIL 환경 및 설정
# ============================================================

import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)

from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)


# ------------------------------------------------------------
# 첫 실행은 반드시 True
# smoke test 성공 후 False로 바꾸고 전체 5-fold 실행
# ------------------------------------------------------------
SMOKE_TEST = False

SEED = 42
IMAGE_SIZE = 224
PATIENT_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4

FULL_EPOCHS = 8
FROZEN_BACKBONE_EPOCHS = 2

HEAD_LR = 1e-4
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-4

NUM_WORKERS = 2

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

AMP_ENABLED = DEVICE.type == "cuda"

print("Device:", DEVICE)
print("Smoke test:", SMOKE_TEST)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)


# ------------------------------------------------------------
# 입력 manifest
# ------------------------------------------------------------
MULTIMODAL_SERIES_PATH = (
    FUSION_DIR
    / "00_multimodal_middle_frame_manifest.csv"
)

MULTIMODAL_PATIENT_PATH = (
    FUSION_DIR
    / "01_multimodal_patient_cohort.csv"
)

series_df = pd.read_csv(
    MULTIMODAL_SERIES_PATH
)

patient_df = pd.read_csv(
    MULTIMODAL_PATIENT_PATH
)

series_df["patient_id"] = (
    pd.to_numeric(series_df["patient_id"])
    .astype("int64")
)

patient_df["patient_id"] = (
    pd.to_numeric(patient_df["patient_id"])
    .astype("int64")
)

patient_df["fold"] = (
    pd.to_numeric(patient_df["fold"])
    .astype("int64")
)

patient_df["target"] = (
    pd.to_numeric(patient_df["target"])
    .astype("int64")
)


# ------------------------------------------------------------
# 입력 Gate
# ------------------------------------------------------------
assert len(patient_df) == 303
assert patient_df["patient_id"].nunique() == 303
assert not patient_df["patient_id"].duplicated().any()

assert series_df["patient_id"].nunique() == 303
assert len(series_df) == 1973

assert series_df["file_path"].map(
    lambda value: Path(str(value)).exists()
).all(), "일부 XCA middle frame이 존재하지 않습니다."

assert set(patient_df["fold"].unique()) == {0, 1, 2, 3, 4}
assert set(patient_df["target"].unique()) == {0, 1}


# ------------------------------------------------------------
# 출력 경로
# ------------------------------------------------------------
if SMOKE_TEST:
    XCA_RUN_DIR = (
        FUSION_DIR
        / "_smoke_test"
        / "xca_patient_mil"
    )
    RUN_FOLDS = [0]
    EPOCHS = 1
else:
    XCA_RUN_DIR = (
        FUSION_DIR
        / "xca_patient_mil_5fold"
    )
    RUN_FOLDS = [0, 1, 2, 3, 4]
    EPOCHS = FULL_EPOCHS

CHECKPOINT_DIR = XCA_RUN_DIR / "checkpoints"
PREDICTION_DIR = XCA_RUN_DIR / "predictions"
HISTORY_DIR = XCA_RUN_DIR / "history"

for directory in [
    CHECKPOINT_DIR,
    PREDICTION_DIR,
    HISTORY_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


run_config = {
    "target": "any_definite_over_50",
    "cohort_patients": 303,
    "selected_middle_frames": 1973,
    "outer_folds": RUN_FOLDS,
    "epochs": EPOCHS,
    "frozen_backbone_epochs": (
        0 if SMOKE_TEST
        else FROZEN_BACKBONE_EPOCHS
    ),
    "image_size": IMAGE_SIZE,
    "patient_batch_size": PATIENT_BATCH_SIZE,
    "gradient_accumulation_steps": (
        GRADIENT_ACCUMULATION_STEPS
    ),
    "backbone": "resnet50_imagenet1k_v2",
    "pooling": "attention_mil_with_view_code",
    "clinical_model": "adaboost",
    "fusion_rule": (
        "0.5 * clinical_score + "
        "0.5 * xca_score"
    ),
    "outer_validation_used_for_training": False,
    "test_evaluated": False,
    "seed": SEED,
}

with (
    XCA_RUN_DIR
    / "00_run_config.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        run_config,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Patients:", len(patient_df))
print("Series:", len(series_df))
print("Run folds:", RUN_FOLDS)
print("Epochs:", EPOCHS)
print("Output:", XCA_RUN_DIR)

print("\n✅ XCA MIL 설정 Gate 통과")
print("🔒 TEST 미사용")

# Dataset, attention MIL 모델 및 평가 함수

In [ ]:
# ============================================================
# Cell 6. Patient-bag dataset 및 attention MIL
# ============================================================

train_transform = transforms.Compose([
    transforms.Grayscale(
        num_output_channels=3,
    ),
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
    ),
    transforms.RandomRotation(
        degrees=5,
        fill=0,
    ),
    transforms.RandomApply(
        [
            transforms.ColorJitter(
                brightness=0.10,
                contrast=0.10,
            )
        ],
        p=0.5,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

validation_transform = transforms.Compose([
    transforms.Grayscale(
        num_output_channels=3,
    ),
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


class PatientBagDataset(Dataset):
    def __init__(
        self,
        patient_table,
        series_table,
        transform,
    ):
        self.patient_table = (
            patient_table
            .sort_values("patient_id")
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform

        self.series_groups = {
            int(patient_id): (
                group
                .sort_values(
                    [
                        "view",
                        "series_id",
                    ]
                )
                .reset_index(drop=True)
            )
            for patient_id, group
            in series_table.groupby("patient_id")
        }

        missing = [
            int(patient_id)
            for patient_id
            in self.patient_table["patient_id"]
            if int(patient_id) not in self.series_groups
        ]

        assert not missing, (
            f"XCA series가 없는 환자: {missing}"
        )

    def __len__(self):
        return len(self.patient_table)

    def __getitem__(self, index):
        patient_row = self.patient_table.iloc[index]

        patient_id = int(
            patient_row["patient_id"]
        )

        target = float(
            patient_row["target"]
        )

        series_rows = self.series_groups[
            patient_id
        ]

        images = []
        view_codes = []
        series_ids = []

        for _, row in series_rows.iterrows():
            image_path = Path(
                str(row["file_path"])
            )

            with Image.open(image_path) as image:
                image = image.convert("L")
                image = self.transform(image)

            view = str(
                row["view"]
            ).strip().upper()

            if view == "LEFT":
                view_code = [1.0, 0.0]
            elif view == "RIGHT":
                view_code = [0.0, 1.0]
            else:
                view_code = [0.0, 0.0]

            images.append(image)
            view_codes.append(view_code)
            series_ids.append(
                str(row["series_id"])
            )

        return {
            "patient_id": patient_id,
            "target": target,
            "images": torch.stack(images),
            "view_codes": torch.tensor(
                view_codes,
                dtype=torch.float32,
            ),
            "series_ids": series_ids,
        }


def patient_bag_collate(batch):
    batch_size = len(batch)

    max_series = max(
        item["images"].shape[0]
        for item in batch
    )

    channels, height, width = (
        batch[0]["images"].shape[1:]
    )

    images = torch.zeros(
        (
            batch_size,
            max_series,
            channels,
            height,
            width,
        ),
        dtype=torch.float32,
    )

    view_codes = torch.zeros(
        (
            batch_size,
            max_series,
            2,
        ),
        dtype=torch.float32,
    )

    valid_mask = torch.zeros(
        (
            batch_size,
            max_series,
        ),
        dtype=torch.bool,
    )

    targets = torch.zeros(
        batch_size,
        dtype=torch.float32,
    )

    patient_ids = torch.zeros(
        batch_size,
        dtype=torch.int64,
    )

    series_ids = []

    for batch_index, item in enumerate(batch):
        series_count = item["images"].shape[0]

        images[
            batch_index,
            :series_count,
        ] = item["images"]

        view_codes[
            batch_index,
            :series_count,
        ] = item["view_codes"]

        valid_mask[
            batch_index,
            :series_count,
        ] = True

        targets[batch_index] = item["target"]
        patient_ids[batch_index] = item["patient_id"]
        series_ids.append(item["series_ids"])

    return {
        "images": images,
        "view_codes": view_codes,
        "valid_mask": valid_mask,
        "targets": targets,
        "patient_ids": patient_ids,
        "series_ids": series_ids,
    }


class PatientAttentionMIL(nn.Module):
    def __init__(
        self,
        hidden_dim=256,
        dropout=0.30,
    ):
        super().__init__()

        weights = (
            ResNet50_Weights.IMAGENET1K_V2
        )

        self.backbone = models.resnet50(
            weights=weights,
        )

        feature_dim = (
            self.backbone.fc.in_features
        )

        self.backbone.fc = nn.Identity()

        self.projection = nn.Sequential(
            nn.Linear(
                feature_dim + 2,
                hidden_dim,
            ),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )

        self.attention = nn.Sequential(
            nn.Linear(
                hidden_dim,
                128,
            ),
            nn.Tanh(),
            nn.Linear(
                128,
                1,
            ),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(
                hidden_dim,
                1,
            ),
        )

    def forward(
        self,
        images,
        view_codes,
        valid_mask,
    ):
        batch_size, max_series = (
            images.shape[:2]
        )

        valid_images = images[
            valid_mask
        ]

        valid_view_codes = view_codes[
            valid_mask
        ]

        image_features = self.backbone(
            valid_images
        )

        combined_features = torch.cat(
            [
                image_features,
                valid_view_codes,
            ],
            dim=1,
        )

        hidden_features = self.projection(
            combined_features
        )

        padded_hidden = (
            hidden_features.new_zeros(
                (
                    batch_size,
                    max_series,
                    hidden_features.shape[-1],
                )
            )
        )

        padded_hidden[
            valid_mask
        ] = hidden_features

        attention_logits = (
            self.attention(
                padded_hidden
            )
            .squeeze(-1)
        )

        attention_logits = (
            attention_logits.masked_fill(
                ~valid_mask,
                torch.finfo(
                    attention_logits.dtype
                ).min,
            )
        )

        attention_weights = torch.softmax(
            attention_logits,
            dim=1,
        )

        patient_feature = (
            padded_hidden
            * attention_weights.unsqueeze(-1)
        ).sum(dim=1)

        patient_logit = (
            self.classifier(
                patient_feature
            )
            .squeeze(1)
        )

        return (
            patient_logit,
            attention_weights,
        )


def set_backbone_trainability(
    model,
    fine_tune_layer4,
):
    for parameter in (
        model.backbone.parameters()
    ):
        parameter.requires_grad = False

    if fine_tune_layer4:
        for parameter in (
            model.backbone
            .layer4
            .parameters()
        ):
            parameter.requires_grad = True


def binary_metrics(
    targets,
    probabilities,
    threshold=0.5,
):
    targets = np.asarray(
        targets,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        targets,
        predictions,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "accuracy": accuracy_score(
            targets,
            predictions,
        ),
        "precision": precision_score(
            targets,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            targets,
            predictions,
            zero_division=0,
        ),
        "specificity": specificity,
        "f1": f1_score(
            targets,
            predictions,
            zero_division=0,
        ),
        "balanced_accuracy": (
            balanced_accuracy_score(
                targets,
                predictions,
            )
        ),
        "roc_auc": roc_auc_score(
            targets,
            probabilities,
        ),
        "pr_auc": average_precision_score(
            targets,
            probabilities,
        ),
    }


print("✅ Dataset/MIL model 정의 완료")

# 누수 없는 5-fold XCA OOF 학습

In [ ]:
# ============================================================
# Cell 7. Patient-level XCA MIL 5-fold OOF
#
# 중요:
# - outer validation fold는 마지막 평가 때만 사용
# - validation 기반 early stopping 없음
# - 고정 epoch의 최종 모델을 사용
# ============================================================

def build_weighted_sampler(
    patient_table,
    seed,
):
    counts = (
        patient_table["target"]
        .value_counts()
        .to_dict()
    )

    sample_weights = (
        patient_table["target"]
        .map(
            lambda target: (
                1.0 / counts[int(target)]
            )
        )
        .to_numpy()
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    return WeightedRandomSampler(
        weights=torch.as_tensor(
            sample_weights,
            dtype=torch.double,
        ),
        num_samples=len(sample_weights),
        replacement=True,
        generator=generator,
    )


def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    fine_tune_layer4,
):
    model.train()

    # 동결된 backbone의 BatchNorm 통계도 고정
    model.backbone.eval()

    if fine_tune_layer4:
        model.backbone.layer4.train()

    optimizer.zero_grad(
        set_to_none=True,
    )

    running_loss = 0.0
    patient_count = 0

    for step, batch in enumerate(loader):
        images = batch["images"].to(
            DEVICE,
            non_blocking=True,
        )

        view_codes = batch[
            "view_codes"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        valid_mask = batch[
            "valid_mask"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        targets = batch["targets"].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.cuda.amp.autocast(
            enabled=AMP_ENABLED,
        ):
            logits, _ = model(
                images,
                view_codes,
                valid_mask,
            )

            raw_loss = criterion(
                logits,
                targets,
            )

            loss = (
                raw_loss
                / GRADIENT_ACCUMULATION_STEPS
            )

        scaler.scale(loss).backward()

        should_step = (
            (step + 1)
            % GRADIENT_ACCUMULATION_STEPS
            == 0
            or (step + 1) == len(loader)
        )

        if should_step:
            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(
                set_to_none=True,
            )

        batch_size = targets.shape[0]

        running_loss += (
            raw_loss.item()
            * batch_size
        )

        patient_count += batch_size

    return running_loss / patient_count


@torch.no_grad()
def predict_patient_loader(
    model,
    loader,
):
    model.eval()

    patient_ids = []
    targets = []
    probabilities = []
    attention_records = []

    for batch in loader:
        images = batch["images"].to(
            DEVICE,
            non_blocking=True,
        )

        view_codes = batch[
            "view_codes"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        valid_mask = batch[
            "valid_mask"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.cuda.amp.autocast(
            enabled=AMP_ENABLED,
        ):
            logits, attention_weights = model(
                images,
                view_codes,
                valid_mask,
            )

        batch_probabilities = (
            torch.sigmoid(logits)
            .cpu()
            .numpy()
        )

        batch_targets = (
            batch["targets"]
            .cpu()
            .numpy()
        )

        batch_patient_ids = (
            batch["patient_ids"]
            .cpu()
            .numpy()
        )

        for index, patient_id in enumerate(
            batch_patient_ids
        ):
            valid_count = int(
                batch["valid_mask"][
                    index
                ].sum()
            )

            attention = (
                attention_weights[
                    index,
                    :valid_count,
                ]
                .float()
                .cpu()
                .numpy()
            )

            patient_ids.append(
                int(patient_id)
            )

            targets.append(
                int(batch_targets[index])
            )

            probabilities.append(
                float(
                    batch_probabilities[index]
                )
            )

            attention_records.append(
                attention.tolist()
            )

    return pd.DataFrame({
        "patient_id": patient_ids,
        "target": targets,
        "xca_probability": probabilities,
        "xca_attention_weights": [
            json.dumps(values)
            for values in attention_records
        ],
    })


fold_metrics = []

for outer_fold in RUN_FOLDS:
    print(
        "\n"
        + "=" * 60
    )
    print(f"OUTER FOLD {outer_fold}")
    print("=" * 60)

    fold_seed = SEED + outer_fold
    set_seed(fold_seed)

    fold_prediction_path = (
        PREDICTION_DIR
        / f"fold_{outer_fold}_oof.csv"
    )

    fold_checkpoint_path = (
        CHECKPOINT_DIR
        / f"fold_{outer_fold}_final.pt"
    )

    # 재실행 시 완료된 fold는 건너뜀
    if (
        fold_prediction_path.exists()
        and fold_checkpoint_path.exists()
    ):
        print(
            "✅ 기존 완료 산출물을 사용합니다:",
            fold_prediction_path,
        )

        existing_fold = pd.read_csv(
            fold_prediction_path
        )

        metrics = binary_metrics(
            existing_fold["target"],
            existing_fold[
                "xca_probability"
            ],
        )

        metrics["fold"] = outer_fold
        fold_metrics.append(metrics)
        continue

    train_patients = (
        patient_df[
            patient_df["fold"]
            != outer_fold
        ]
        .sort_values("patient_id")
        .reset_index(drop=True)
    )

    validation_patients = (
        patient_df[
            patient_df["fold"]
            == outer_fold
        ]
        .sort_values("patient_id")
        .reset_index(drop=True)
    )

    train_ids = set(
        train_patients["patient_id"]
    )

    validation_ids = set(
        validation_patients["patient_id"]
    )

    assert train_ids.isdisjoint(
        validation_ids
    )

    train_series = series_df[
        series_df["patient_id"].isin(
            train_ids
        )
    ].copy()

    validation_series = series_df[
        series_df["patient_id"].isin(
            validation_ids
        )
    ].copy()

    print(
        "Train patients:",
        len(train_patients),
    )

    print(
        "Validation patients:",
        len(validation_patients),
    )

    print(
        "Train target:",
        train_patients["target"]
        .value_counts()
        .sort_index()
        .to_dict(),
    )

    print(
        "Validation target:",
        validation_patients["target"]
        .value_counts()
        .sort_index()
        .to_dict(),
    )

    train_dataset = PatientBagDataset(
        train_patients,
        train_series,
        train_transform,
    )

    validation_dataset = PatientBagDataset(
        validation_patients,
        validation_series,
        validation_transform,
    )

    sampler = build_weighted_sampler(
        train_patients,
        fold_seed,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=PATIENT_BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=AMP_ENABLED,
        collate_fn=patient_bag_collate,
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=PATIENT_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=AMP_ENABLED,
        collate_fn=patient_bag_collate,
    )

    model = PatientAttentionMIL().to(
        DEVICE
    )

    set_backbone_trainability(
        model,
        fine_tune_layer4=False,
    )

    head_parameters = list(
        model.projection.parameters()
    ) + list(
        model.attention.parameters()
    ) + list(
        model.classifier.parameters()
    )

    optimizer = torch.optim.AdamW(
        [
            {
                "params": (
                    model.backbone.parameters()
                ),
                "lr": BACKBONE_LR,
            },
            {
                "params": head_parameters,
                "lr": HEAD_LR,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=EPOCHS,
        )
    )

    criterion = (
        nn.BCEWithLogitsLoss()
    )

    scaler = (
        torch.cuda.amp.GradScaler(
            enabled=AMP_ENABLED,
        )
    )

    history_rows = []

    for epoch in range(EPOCHS):
        fine_tune_layer4 = (
            not SMOKE_TEST
            and epoch
            >= FROZEN_BACKBONE_EPOCHS
        )

        set_backbone_trainability(
            model,
            fine_tune_layer4,
        )

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            fine_tune_layer4=(
                fine_tune_layer4
            ),
        )

        scheduler.step()

        history_rows.append({
            "fold": outer_fold,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "layer4_trainable": (
                fine_tune_layer4
            ),
            "backbone_lr": (
                optimizer.param_groups[0][
                    "lr"
                ]
            ),
            "head_lr": (
                optimizer.param_groups[1][
                    "lr"
                ]
            ),
        })

        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} "
            f"| train_loss={train_loss:.4f} "
            f"| layer4={fine_tune_layer4}"
        )

    # outer validation은 학습 종료 후 한 번만 평가
    fold_predictions = (
        predict_patient_loader(
            model,
            validation_loader,
        )
    )

    fold_predictions["fold"] = (
        outer_fold
    )

    fold_predictions[
        "xca_prediction"
    ] = (
        fold_predictions[
            "xca_probability"
        ] >= 0.5
    ).astype(int)

    expected_validation_ids = set(
        validation_patients[
            "patient_id"
        ]
    )

    predicted_validation_ids = set(
        fold_predictions[
            "patient_id"
        ]
    )

    assert (
        expected_validation_ids
        == predicted_validation_ids
    )

    assert len(fold_predictions) == len(
        validation_patients
    )

    metrics = binary_metrics(
        fold_predictions["target"],
        fold_predictions[
            "xca_probability"
        ],
    )

    metrics["fold"] = outer_fold
    fold_metrics.append(metrics)

    fold_predictions.to_csv(
        fold_prediction_path,
        index=False,
    )

    pd.DataFrame(
        history_rows
    ).to_csv(
        HISTORY_DIR
        / f"fold_{outer_fold}_history.csv",
        index=False,
    )

    torch.save(
        {
            "fold": outer_fold,
            "epoch_count": EPOCHS,
            "model_state_dict": (
                model.state_dict()
            ),
            "run_config": run_config,
        },
        fold_checkpoint_path,
    )

    print(
        "\nFold metrics:",
        {
            key: round(value, 4)
            for key, value in metrics.items()
            if key != "fold"
        },
    )

    del (
        model,
        optimizer,
        scheduler,
        scaler,
        train_loader,
        validation_loader,
        train_dataset,
        validation_dataset,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


fold_metrics_df = pd.DataFrame(
    fold_metrics
).sort_values("fold")

display(fold_metrics_df)

fold_metrics_df.to_csv(
    XCA_RUN_DIR
    / "01_fold_metrics.csv",
    index=False,
)

print("\n✅ 지정된 XCA fold 학습 완료")
print("🔒 TEST 미사용")

# 전체 XCA OOF 및 Clinical 1:1 fusion

In [ ]:
# ============================================================
# Cell 8. XCA OOF 결합 및 Clinical+XCA late fusion
# ============================================================

assert not SMOKE_TEST, (
    "전체 fusion 평가는 "
    "SMOKE_TEST=False의 5-fold 완료 후 실행합니다."
)

fold_prediction_paths = [
    PREDICTION_DIR
    / f"fold_{fold}_oof.csv"
    for fold in range(5)
]

missing_fold_files = [
    path
    for path in fold_prediction_paths
    if not path.exists()
]

assert not missing_fold_files, (
    f"완료되지 않은 fold가 있습니다: {missing_fold_files}"
)


# ------------------------------------------------------------
# 1. XCA 5-fold OOF
# ------------------------------------------------------------
xca_oof = pd.concat(
    [
        pd.read_csv(path)
        for path in fold_prediction_paths
    ],
    ignore_index=True,
)

xca_oof["patient_id"] = (
    pd.to_numeric(
        xca_oof["patient_id"]
    ).astype("int64")
)

xca_oof["fold"] = (
    pd.to_numeric(
        xca_oof["fold"]
    ).astype("int64")
)

assert len(xca_oof) == 303
assert xca_oof["patient_id"].nunique() == 303
assert not xca_oof["patient_id"].duplicated().any()
assert xca_oof["xca_probability"].between(0, 1).all()

xca_oof.to_csv(
    FUSION_DIR
    / "02_xca_patient_oof.csv",
    index=False,
)


# ------------------------------------------------------------
# 2. AdaBoost Clinical OOF
# ------------------------------------------------------------
clinical_oof = pd.read_csv(
    CLINICAL_OOF_PATH
)

clinical_primary = (
    clinical_oof[
        clinical_oof["model"].eq(
            "adaboost"
        )
    ]
    [
        [
            "patient_id",
            "fold",
            "target",
            "probability",
            "prediction",
        ]
    ]
    .rename(
        columns={
            "probability": (
                "clinical_probability"
            ),
            "prediction": (
                "clinical_prediction"
            ),
        }
    )
    .copy()
)

clinical_primary["patient_id"] = (
    pd.to_numeric(
        clinical_primary["patient_id"]
    ).astype("int64")
)

clinical_primary["fold"] = (
    pd.to_numeric(
        clinical_primary["fold"]
    ).astype("int64")
)


# ------------------------------------------------------------
# 3. 동일 patient/fold 결합
# ------------------------------------------------------------
fusion_oof = clinical_primary.merge(
    xca_oof[
        [
            "patient_id",
            "fold",
            "target",
            "xca_probability",
            "xca_prediction",
        ]
    ],
    on=[
        "patient_id",
        "fold",
    ],
    how="inner",
    suffixes=(
        "_clinical",
        "_xca",
    ),
    validate="one_to_one",
)

assert len(fusion_oof) == 303
assert fusion_oof["patient_id"].nunique() == 303

assert (
    fusion_oof[
        "target_clinical"
    ]
    == fusion_oof[
        "target_xca"
    ]
).all()

fusion_oof["target"] = (
    fusion_oof[
        "target_clinical"
    ]
)

fusion_oof = fusion_oof.drop(
    columns=[
        "target_clinical",
        "target_xca",
    ]
)


# ------------------------------------------------------------
# 4. 사전 고정 1:1 late fusion
# ------------------------------------------------------------
fusion_oof[
    "fusion_probability"
] = (
    0.5
    * fusion_oof[
        "clinical_probability"
    ]
    + 0.5
    * fusion_oof[
        "xca_probability"
    ]
)

fusion_oof[
    "fusion_prediction"
] = (
    fusion_oof[
        "fusion_probability"
    ] >= 0.5
).astype(int)


# ------------------------------------------------------------
# 5. 세 모델 비교
# ------------------------------------------------------------
model_scores = {
    "clinical_adaboost": (
        "clinical_probability"
    ),
    "xca_patient_mil": (
        "xca_probability"
    ),
    "clinical_xca_equal_fusion": (
        "fusion_probability"
    ),
}

metric_rows = []

for model_name, score_column in (
    model_scores.items()
):
    metrics = binary_metrics(
        fusion_oof["target"],
        fusion_oof[score_column],
        threshold=0.5,
    )

    metrics["model"] = model_name
    metric_rows.append(metrics)

metrics_df = (
    pd.DataFrame(metric_rows)
    [
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "specificity",
            "f1",
            "balanced_accuracy",
            "roc_auc",
            "pr_auc",
        ]
    ]
    .sort_values(
        "balanced_accuracy",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("=== DEVELOPMENT 303명 전체 OOF ===")
display(metrics_df.round(4))


# ------------------------------------------------------------
# 6. Fold별 비교
# ------------------------------------------------------------
fold_metric_rows = []

for fold in range(5):
    fold_data = fusion_oof[
        fusion_oof["fold"] == fold
    ]

    for model_name, score_column in (
        model_scores.items()
    ):
        metrics = binary_metrics(
            fold_data["target"],
            fold_data[score_column],
            threshold=0.5,
        )

        metrics["fold"] = fold
        metrics["model"] = model_name

        fold_metric_rows.append(metrics)

fold_metrics_df = pd.DataFrame(
    fold_metric_rows
)

print("\n=== Fold별 성능 ===")
display(
    fold_metrics_df[
        [
            "fold",
            "model",
            "balanced_accuracy",
            "roc_auc",
            "pr_auc",
            "specificity",
            "recall",
        ]
    ].round(4)
)


# ------------------------------------------------------------
# 7. 저장
# ------------------------------------------------------------
fusion_oof.to_csv(
    FUSION_DIR
    / "03_fusion_oof_predictions.csv",
    index=False,
)

metrics_df.to_csv(
    FUSION_DIR
    / "04_fusion_oof_metrics.csv",
    index=False,
)

fold_metrics_df.to_csv(
    FUSION_DIR
    / "05_fusion_fold_metrics.csv",
    index=False,
)

print(
    "\n저장:",
    FUSION_DIR
    / "03_fusion_oof_predictions.csv",
)

print(
    "저장:",
    FUSION_DIR
    / "04_fusion_oof_metrics.csv",
)

print(
    "저장:",
    FUSION_DIR
    / "05_fusion_fold_metrics.csv",
)

print("\n✅ Clinical/XCA/Fusion OOF 비교 완료")
print("✅ Fusion 가중치는 1:1로 사전 고정")
print("🔒 TEST는 평가하지 않았습니다.")

In [ ]:
# ============================================================
# Cell 9. 환자단위 paired stratified bootstrap
# - fold × target 비율을 유지한 paired resampling
# - Fusion 개선폭의 95% CI 계산
# - TEST 미사용
# ============================================================

from scipy.stats import spearmanr

BOOTSTRAP_ITERATIONS = 5000
BOOTSTRAP_SEED = 20260902

fusion_oof = pd.read_csv(
    FUSION_DIR
    / "03_fusion_oof_predictions.csv"
)

fusion_oof["patient_id"] = pd.to_numeric(
    fusion_oof["patient_id"]
).astype("int64")

fusion_oof["fold"] = pd.to_numeric(
    fusion_oof["fold"]
).astype("int64")

fusion_oof["target"] = pd.to_numeric(
    fusion_oof["target"]
).astype("int64")

assert len(fusion_oof) == 303
assert fusion_oof["patient_id"].nunique() == 303


score_columns = {
    "clinical_adaboost": "clinical_probability",
    "xca_patient_mil": "xca_probability",
    "clinical_xca_equal_fusion": "fusion_probability",
}

metric_names = [
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "balanced_accuracy",
    "roc_auc",
    "pr_auc",
]


# ------------------------------------------------------------
# 1. Point estimates
# ------------------------------------------------------------
point_metrics = {}

for model_name, score_column in score_columns.items():
    point_metrics[model_name] = binary_metrics(
        fusion_oof["target"],
        fusion_oof[score_column],
        threshold=0.5,
    )


# ------------------------------------------------------------
# 2. fold × target strata
# ------------------------------------------------------------
strata_indices = {
    (int(fold), int(target)): group.index.to_numpy()
    for (fold, target), group
    in fusion_oof.groupby(["fold", "target"])
}

print("Bootstrap strata:")

for key, indices in strata_indices.items():
    print(key, len(indices))


# ------------------------------------------------------------
# 3. Paired bootstrap
# ------------------------------------------------------------
rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_records = []

for iteration in range(
    BOOTSTRAP_ITERATIONS
):
    sampled_indices = []

    for indices in strata_indices.values():
        sampled_indices.extend(
            rng.choice(
                indices,
                size=len(indices),
                replace=True,
            )
        )

    sampled = fusion_oof.loc[
        sampled_indices
    ]

    iteration_metrics = {
        model_name: binary_metrics(
            sampled["target"],
            sampled[score_column],
            threshold=0.5,
        )
        for model_name, score_column
        in score_columns.items()
    }

    for model_name in score_columns:
        row = {
            "iteration": iteration,
            "model": model_name,
        }

        row.update(
            iteration_metrics[model_name]
        )

        bootstrap_records.append(row)

bootstrap_df = pd.DataFrame(
    bootstrap_records
)


# ------------------------------------------------------------
# 4. 모델별 metric 95% CI
# ------------------------------------------------------------
ci_rows = []

for model_name in score_columns:
    model_bootstrap = bootstrap_df[
        bootstrap_df["model"] == model_name
    ]

    for metric_name in metric_names:
        values = model_bootstrap[
            metric_name
        ].to_numpy()

        ci_rows.append({
            "model": model_name,
            "metric": metric_name,
            "point_estimate": (
                point_metrics[
                    model_name
                ][metric_name]
            ),
            "ci_95_lower": np.quantile(
                values,
                0.025,
            ),
            "ci_95_upper": np.quantile(
                values,
                0.975,
            ),
        })

metric_ci_df = pd.DataFrame(ci_rows)

print("\n=== Model metric 95% CI ===")

display(
    metric_ci_df[
        metric_ci_df["metric"].isin(
            [
                "balanced_accuracy",
                "roc_auc",
                "pr_auc",
                "specificity",
                "recall",
            ]
        )
    ].round(4)
)


# ------------------------------------------------------------
# 5. Paired delta CI
# ------------------------------------------------------------
delta_comparisons = [
    (
        "fusion_minus_clinical",
        "clinical_xca_equal_fusion",
        "clinical_adaboost",
    ),
    (
        "fusion_minus_xca",
        "clinical_xca_equal_fusion",
        "xca_patient_mil",
    ),
]

delta_rows = []

for (
    comparison_name,
    first_model,
    second_model,
) in delta_comparisons:

    first_bootstrap = (
        bootstrap_df[
            bootstrap_df["model"]
            == first_model
        ]
        .sort_values("iteration")
        .reset_index(drop=True)
    )

    second_bootstrap = (
        bootstrap_df[
            bootstrap_df["model"]
            == second_model
        ]
        .sort_values("iteration")
        .reset_index(drop=True)
    )

    assert (
        first_bootstrap["iteration"]
        == second_bootstrap["iteration"]
    ).all()

    for metric_name in metric_names:
        delta_values = (
            first_bootstrap[metric_name]
            - second_bootstrap[metric_name]
        ).to_numpy()

        point_delta = (
            point_metrics[first_model][
                metric_name
            ]
            - point_metrics[second_model][
                metric_name
            ]
        )

        delta_rows.append({
            "comparison": comparison_name,
            "metric": metric_name,
            "point_delta": point_delta,
            "ci_95_lower": np.quantile(
                delta_values,
                0.025,
            ),
            "ci_95_upper": np.quantile(
                delta_values,
                0.975,
            ),
            "bootstrap_probability_delta_gt_0": (
                delta_values > 0
            ).mean(),
        })

delta_ci_df = pd.DataFrame(
    delta_rows
)

print("\n=== Paired delta 95% CI ===")

display(
    delta_ci_df[
        delta_ci_df["metric"].isin(
            [
                "balanced_accuracy",
                "roc_auc",
                "pr_auc",
                "specificity",
                "recall",
            ]
        )
    ].round(4)
)


# ------------------------------------------------------------
# 6. Clinical/XCA score 상관성
# ------------------------------------------------------------
spearman_result = spearmanr(
    fusion_oof[
        "clinical_probability"
    ],
    fusion_oof[
        "xca_probability"
    ],
)

correlation_df = pd.DataFrame([{
    "method": "spearman",
    "correlation": (
        spearman_result.statistic
    ),
    "p_value": (
        spearman_result.pvalue
    ),
}])

print("\n=== Clinical-XCA score correlation ===")
display(correlation_df.round(4))


# ------------------------------------------------------------
# 7. 고정 threshold에서 오류 상보성
# ------------------------------------------------------------
target = fusion_oof[
    "target"
].to_numpy()

clinical_prediction = (
    fusion_oof[
        "clinical_probability"
    ].to_numpy() >= 0.5
).astype(int)

xca_prediction = (
    fusion_oof[
        "xca_probability"
    ].to_numpy() >= 0.5
).astype(int)

fusion_prediction = (
    fusion_oof[
        "fusion_probability"
    ].to_numpy() >= 0.5
).astype(int)

clinical_correct = (
    clinical_prediction == target
)

xca_correct = (
    xca_prediction == target
)

fusion_correct = (
    fusion_prediction == target
)

complementarity_df = pd.DataFrame([
    {
        "category": "both_clinical_xca_correct",
        "patient_count": int(
            (
                clinical_correct
                & xca_correct
            ).sum()
        ),
    },
    {
        "category": "clinical_only_correct",
        "patient_count": int(
            (
                clinical_correct
                & ~xca_correct
            ).sum()
        ),
    },
    {
        "category": "xca_only_correct",
        "patient_count": int(
            (
                ~clinical_correct
                & xca_correct
            ).sum()
        ),
    },
    {
        "category": "both_clinical_xca_wrong",
        "patient_count": int(
            (
                ~clinical_correct
                & ~xca_correct
            ).sum()
        ),
    },
    {
        "category": "fusion_correct",
        "patient_count": int(
            fusion_correct.sum()
        ),
    },
])

print("\n=== Error complementarity ===")
display(complementarity_df)


# ------------------------------------------------------------
# 8. 저장
# ------------------------------------------------------------
metric_ci_df.to_csv(
    FUSION_DIR
    / "06_bootstrap_metric_ci.csv",
    index=False,
)

delta_ci_df.to_csv(
    FUSION_DIR
    / "07_bootstrap_delta_ci.csv",
    index=False,
)

correlation_df.to_csv(
    FUSION_DIR
    / "08_clinical_xca_score_correlation.csv",
    index=False,
)

complementarity_df.to_csv(
    FUSION_DIR
    / "09_error_complementarity.csv",
    index=False,
)

print("\n✅ Paired bootstrap 완료")
print("✅ Fusion 개선폭의 95% CI 저장 완료")
print("🔒 TEST 미사용")